In [ ]:
!pip install faiss-cpu


In [ ]:
!pip install tabula-py pandas sentence-transformers google-generativeai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 40.3 MB/s eta 0:00:00


In [ ]:
path='/Orders.xlsx'


In [ ]:
import tabula
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import google.generativeai as genai
from google.colab import userdata

In [ ]:
Gemini_API_Key=userdata.get('Gemini_API_Key')
if Gemini_API_Key is None:
  raise ValueError("No Gemini_API_Key is found")
genai.configure(api_key=Gemini_API_Key)

In [ ]:

df=pd.read_excel(path)
print(df)

    order_id     order_status              customer order_date  \
0          3  Order\rFinished  Muhammed Mac\rIntyre 2010-10-13   
1        293  Order\rFinished          Barry French 2012-10-01   
2        483  Order\rFinished         Clay Rozendal 2011-07-10   
3        515  Order\rFinished        Carlos Soltero 2010-08-28   
4        613  Order\rFinished          Carl Jackson 2011-06-17   
5        643  Order\rFinished        Monica Federle 2011-03-24   
6        678  Order\rReturned       Dorothy Badders 2010-02-26   
7        807  Order\rFinished       Neola Schneider 2010-11-23   
8        868  Order\rFinished           Carlos Daly 2012-06-08   
9        933  Order\rFinished         Claudia Miner 2012-08-04   
10       995  Order\rFinished       Neola Schneider 2011-05-30   
11       998  Order\rFinished      Allen Rosenblatt 2009-11-25   
12      1154  Order\rFinished       Sylvia Foulston 2012-02-14   
13      1344  Order\rFinished           Jim Radford 2012-04-15   
14      14

In [ ]:
meta_data=df.describe(include='all').fillna('')
display(meta_data)

,order_id,order_status,customer,order_date,order_quantity,sales
count,20.0,20,20,20,20.0,20.0
unique,,2,16,,,
top,,Order\rFinished,Carlos Soltero,,,
freq,,19,3,,,
mean,1003.65,,,2011-05-14 12:00:00,27.3,4032125.35
min,3.0,,,2009-11-25 00:00:00,6.0,118060.0
25%,635.5,,,2010-11-01 12:00:00,15.75,342045.0
50%,964.0,,,2011-04-14 12:00:00,26.5,764750.0
75%,1443.75,,,2012-02-29 06:00:00,35.75,4114145.0
max,1792.0,,,2012-10-01 00:00:00,49.0,24056460.0


In [ ]:
documents = []
for _, row in df.iterrows():
  doc = (
      f"Order ID: {row['order_id']} has status {row['order_status']}\n"
      f"Customer is {row['customer']}."
      f"Order Date is {row['order_date']}."
      f"Quantity Ordered is {row['order_quantity']}."
      f"Total Sales amount is {row['sales']}."
  )
  documents.append(doc)

print(documents)

['Order ID: 3 has status Order\\rFinished\nCustomer is Muhammed Mac\\rIntyre.Order Date is 2010-10-13 00:00:00.Quantity Ordered is 6.Total Sales amount is 523080.', 'Order ID: 293 has status Order\\rFinished\nCustomer is Barry French.Order Date is 2012-10-01 00:00:00.Quantity Ordered is 49.Total Sales amount is 20246040.', 'Order ID: 483 has status Order\\rFinished\nCustomer is Clay Rozendal.Order Date is 2011-07-10 00:00:00.Quantity Ordered is 30.Total Sales amount is 9931519.', 'Order ID: 515 has status Order\\rFinished\nCustomer is Carlos Soltero.Order Date is 2010-08-28 00:00:00.Quantity Ordered is 19.Total Sales amount is 788540.', 'Order ID: 613 has status Order\\rFinished\nCustomer is Carl Jackson.Order Date is 2011-06-17 00:00:00.Quantity Ordered is 12.Total Sales amount is 187080.', 'Order ID: 643 has status Order\\rFinished\nCustomer is Monica Federle.Order Date is 2011-03-24 00:00:00.Quantity Ordered is 21.Total Sales amount is 5563640.', 'Order ID: 678 has status Order\\rRe

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#embeddings
model=SentenceTransformer('sentence-transformers/all-miniLM-L6-v2')
embeddings=model.encode(documents,convert_to_numpy=True,show_progress_bar=True)
print(embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[[-0.0531115  -0.0253893   0.0304186  ... -0.03634322 -0.046556
   0.03768994]
 [-0.04064846 -0.00404917 -0.01074561 ... -0.02884728 -0.06390351
  -0.0145968 ]
 [-0.0597324  -0.01271612 -0.00633057 ... -0.00966161 -0.07273875
  -0.01768629]
 ...
 [ 0.00922527  0.0265976   0.05770034 ... -0.00990441 -0.01055641
  -0.02974943]
 [-0.08787361  0.03158558  0.02962808 ... -0.02135037 -0.01379733
   0.00251404]
 [-0.06940062  0.01317554 -0.00728172 ...  0.01527077 -0.08632758
   0.00168001]]


In [ ]:
dimension=embeddings.shape[1]
index=faiss.IndexFlatL2(dimension)
index.add(embeddings)
faiss.write_index(index,'faiss_index.faiss')

In [ ]:
def retrieve_context(query,k=3):
  query_embeddings=model.encode([query])
  distance,indices=index.search(query_embeddings,k)
  return '\n'.join([documents[i] for i in indices[0]])


In [ ]:
generation_config={'temperature':0.4,'max_output_tokens':512}

In [ ]:
gemini_model=genai.GenerativeModel(model_name='models/gemini-2.5-flash',generation_config=generation_config)

In [ ]:
chat_history=[]
def chat_with_bot(user_input):
  global chat_history
  context= retrieve_context(user_input)
  prompt=f""" you are a conversational data analyst chatbot. please answer the user question using the given context below.
              context={context}
              user_question={user_input}
              Rules:
              1.Be conversational
              2.Only answer using the given context
              3.Say don't have enough imformation, if you don't know the answer."""
  response=gemini_model.generate_content(prompt)
  answer=response.text
  chat_history.append({'user':user_input,'bot':answer})
  return answer


In [ ]:
print('Conversational chatbot is ready for sales document')
print("type exit to stop the conversation \n")
while True:
  user_input=input("User:")
  if user_input.lower() in ['exit','stop','bye']:
    print("Good Bye!")
    break
  response=chat_with_bot(user_input)
  print(f'Bot:{response}')
  print('_'*60)

Conversational chatbot is ready for sales document
type exit to stop the conversation 

User:status of order 293
Bot:Hello there!

Looking at the data, Order ID 293 has a status of **Order Finished**.

Is there anything else I can help you with regarding this order or others?
____________________________________________________________
User:exit
Good Bye!
